# z-shift Tier B - GPU experiments

Everything in the protocol that needs CUDA: **B1-B8**. Tier A is CPU-only and is
not run here.

Runs on **Kaggle** (recommended) or **Colab**. Kaggle is preferred because
*Save & Run All* executes headless for up to 12 h and survives a closed browser,
and because a private Kaggle Dataset mounts your DTU / capture data read-only at
`/kaggle/input` every session with no re-download.

## Read this before you press Run All

1. **`--quick` is a no-op in Tier B.** `experiment_parser` defines the flag and
   not one of the eight `exp_b*.py` modules forwards it to `run()`. Typing the
   smoke-test command in a terminal silently launches the full grid. This
   notebook therefore never passes `--quick`; scope is controlled by the flags
   that are actually honoured (`--scenes`, `--n-images`, `--frame-counts`,
   `--thresholds`, `--no-budget-sweep`). Wiring `--quick` properly is still
   TODO P0 and worth doing.
2. **Run the SMOKE lane first.** It exercises every module end to end in
   minutes. A three-day run that dies at row 1 on a `KeyError` is the failure
   mode this exists to prevent.
3. **The repo is cloned from GitHub.** `bench/`, `paper/`,
   `instrumentation.py` and `tests/test_bench.py` are untracked on your laptop.
   Commit and push before running this, or the clone will not contain them.
4. **`data/` and `third_party/` are gitignored.** MASt3R is cloned and installed
   by this notebook; the dataset must come from a mounted Kaggle Dataset or
   Drive folder that you point `DATA_ROOT` at.

5. **The dataset test (section 10) runs before anything expensive.** It reads
   the Kaggle datasets you actually mounted rather than a hardcoded slug, and
   fails in seconds on what would otherwise fail three hours in: an unreadable
   image, a ground-truth file trimesh cannot open, a `tau` stated in the wrong
   unit, a scene with no ground truth at all.

## What "proper error logs" means here

- One log file per experiment under `<results>/logs/<run_id>/`, plus a combined
  `session.log` holding every log record from the notebook itself.
- Each experiment runs in its **own subprocess**, so a CUDA OOM or a VTK
  segfault kills that experiment and not the kernel. Exit code, signal, wall
  time and the tail of the output are recorded either way.
- `faulthandler` is enabled in parent and children, so a native crash leaves a
  Python traceback instead of a silent death.
- `run_summary.json` is rewritten after **every** experiment, so a session
  killed at the 12 h wall still leaves a full account of what finished.
- Re-running the notebook **skips experiments whose CSV already exists**, so a
  killed session resumes rather than restarts.

## How to run this, and what it costs

**Kaggle is the right platform for the full lane.** *Save & Run All* executes
headless for up to 12 h and survives a closed browser; Colab free disconnects on
idle after ~90 minutes and needs the tab open, which makes it a smoke-lane tool.

### Kaggle

1. **New Notebook > File > Import Notebook**, upload `notebooks/tier_b_gpu.ipynb`.
2. **Settings > Accelerator > GPU T4 x2** (or P100), **Internet > On**. Internet
   is not optional: `pip`, the MASt3R clone and the 2.6 GB checkpoint all need it.
3. **Input > Add Input**, paste each slug:
   - `jinnywjy/tanks-and-temple-m60-colmap-preprocessed`
   - `nguyenhung1903/nerf-synthetic-dataset`
4. Leave `SCOPE = "smoke"` for the first run. Read section 10's table before
   anything else — it fails in seconds on data problems that otherwise surface
   three hours in.
5. **Save Version > Save & Run All (Commit)**. Results land in the Output tab as
   `tier_b_<run_id>.zip`.

**Resuming.** Re-running skips experiments whose CSV already exists, but a fresh
commit starts with an empty `/kaggle/working` — the skip only helps within a
session unless you attach the previous version's **output as an input** and point
`RESULTS_DIR` at it. For the full lane, plan on two sessions and do this between
them.

### Colab

Runtime > Change runtime type > **T4** (free) or **L4/A100** (Pro). There is no
`/kaggle/input`, so either:

- set `FETCH_MISSING_DATASETS = True` and provide a Kaggle API token
  (`~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME` / `KAGGLE_KEY` in the
  environment) — the notebook then pulls both slugs with `kagglehub`; or
- mount Drive, copy the data there, and point `DATA_ROOT` at that folder.

Mount Drive either way, or `OUT_DIR` dies with the runtime.

### What it costs

Estimates, not measurements — scaled from the one number this project has
measured (**62 s per MASt3R pair at 512 px** on its i7-1255U CPU) against
typical T4 throughput of roughly 0.5–0.8 s per pair. An L4 or A100 is about
2–3x faster than the figures below.

| Stage | T4 | Notes |
|---|---|---|
| One-time setup (cells 2–7) | **12–18 min** | pip, MASt3R clone, RoPE CUDA build, 2.6 GB checkpoint |
| Dataset test (sections 8–10) | **< 2 min** | plus the mount walk, seconds |
| **Smoke lane, all 8** | **1.5–2.5 h** | b4 alone is 45–75 min of it: three 40-frame reconstructions, and `DEFAULT_BUDGET` has no CLI override |
| **Full lane, all 8** | **9–15 h** | two Kaggle sessions; 4–7 h on an L4/A100 |

Smoke lane, per experiment: b1 6–10 min (two use cases, 6 frames, and the only
one that emits a rigged GLB), b2 4–6, b6 5–8, b5 6–10, b8 8–12, b7 10–15,
b3 10–15, b4 45–75.

Kaggle's GPU quota is ~30 h/week, so a smoke lane plus a two-session full lane
fits inside one week with room to repeat a failed experiment.

### What to look at first

B1 writes `rigged_mesh.glb` next to `skeleton.json` and `rigging_manifest.json`
under the deliverables root, and it runs first in the smoke lane. Open the GLB in
Blender: the mesh should import with an armature bound to it. With
`ARTICULATION = "static"` that armature is a single root joint — correct for a
tank or a bulldozer, and the thing to change if you point this at a person or an
animal. `b1_end_to_end.csv` records `has_rigged_mesh_path` and the articulation
used, but whether the armature *imports cleanly* is a human check; the CSV says
so in `blender_armature_check` rather than pretending to assert it.

## 1. Config

The only cell you should need to edit.

In [ ]:
# --- repo -----------------------------------------------------------------
REPO_URL = "https://github.com/AyushK0808/z-shift.git"
REPO_BRANCH = "main"

# --- data -----------------------------------------------------------------
# On Kaggle every attached dataset mounts read-only under /kaggle/input, so the
# root is the whole mount area. On Colab point this at a Drive folder, e.g.
# "/content/drive/MyDrive/zshift-tier-b".
DATA_ROOT = "/kaggle/input"

# Datasets to reconstruct, by Kaggle slug. Attach each one in the sidebar
# (Input > Add Input > paste the slug); it mounts at /kaggle/input/<name> and
# costs no working disk.
#
# `image_dir` is relative to the mount and is a *hint*. If it is missing or the
# mirror repackaged the tree, section 8 walks that mount and takes the largest
# image folder it finds, so the scene still resolves.
#
# `tau` is the F-score threshold in the dataset's own units. Neither dataset
# below ships a sensor ground-truth cloud, so B2/B4/B6/B8 have nothing to score
# against and the value only matters once you add one -- see section 10, which
# checks tau against the GT bounding diagonal when a GT is present.
KAGGLE_SCENES = [
    {
        "slug": "jinnywjy/tanks-and-temple-m60-colmap-preprocessed",
        "name": "m60_tank",
        "image_dir": "images",
        "gt_path": None,
        "tau": 0.01,
        "units": "scene units (COLMAP)",
        "notes": (
            "Tanks and Temples M60, ~313 frames, COLMAP-preprocessed. A real "
            "photographed object: the one to eyeball in Blender. Any COLMAP "
            "cloud in the mount is pseudo-GT -- label it as such in the paper."
        ),
    },
    {
        "slug": "nguyenhung1903/nerf-synthetic-dataset",
        "name": "lego",
        "image_dir": "nerf_synthetic/lego/train",
        "gt_path": None,
        "tau": 0.01,
        "units": "scene units (Blender)",
        "notes": (
            "NeRF-synthetic lego bulldozer, 100 views at 800x800, full 360. "
            "Synthetic and unambiguous to verify by eye; the white background "
            "gives MASt3R nothing to match on, so expect a weaker result than "
            "the tank and read it as a control, not a headline."
        ),
    },
]

# Scenes on a mount you point DATA_ROOT at directly, with paths relative to it.
# Left empty because KAGGLE_SCENES covers the attached datasets; fill it in for
# a DTU mount or your own capture:
#
# SCENES = [
#     {
#         "name": "scan24",
#         "image_dir": "dtu/Rectified/scan24",
#         "gt_path": "dtu/Points/stl/stl024_total.ply",
#         "tau": 2.0,
#         "units": "mm",
#         "notes": "DTU eval subset; sensor ground truth",
#     },
# ]
SCENES = []

# --- kaggle plumbing ------------------------------------------------------
# Pull a slug with kagglehub when it is not attached. Attaching is better: it
# mounts instantly, needs no internet and costs no working disk. On Colab there
# is no /kaggle/input at all, so this (plus a Kaggle API token) is the way in.
FETCH_MISSING_DATASETS = False

# If nothing above resolves, walk the mounts and build the scene list from what
# is actually there. Never silent: it logs what it picked, and the dataset test
# then runs against it.
AUTO_DISCOVER_SCENES = True

# Ceiling on auto-discovered scenes -- DTU ships 128 scans.
MAX_DISCOVERED_SCENES = 8

# tau and units for any scene that does not state its own. DTU's convention.
DEFAULT_TAU = 2.0
DEFAULT_UNITS = "mm"

# Stop the notebook when a dataset check FAILS. Warnings never stop it.
DATASET_TEST_STRICT = True

# --- rigging --------------------------------------------------------------
# The skeleton template B1 fits to the refined mesh. Both datasets above are
# objects, so "static": one root joint with every vertex bound to it. A human
# capture wants "biped", an animal "quadruped", a bird "winged". B1 fits one
# template per run -- run it twice for a mixed dataset.
ARTICULATION = "static"

# Inputs per scene for B1's smoke lane. B1 runs the whole pipeline (ingest ->
# MASt3R -> refine -> rig -> deliverable) per use case, so the smoke lane wants
# a handful of frames. The full lane passes everything and lets the pipeline's
# own MAX_RECONSTRUCTION_FRAMES=40 cap decide, which is the behaviour A5/B4
# are about.
B1_SMOKE_IMAGES = 6

# --- scope ----------------------------------------------------------------
# "smoke" = minutes per experiment, validates the orchestration, NOT publishable.
# "full"  = the real grid.
SCOPE = "smoke"

# Which experiments to run, in order. Cheap-and-informative first, so a session
# that dies early still bought you something.
EXPERIMENTS = ["b7", "b2", "b8", "b6", "b3", "b5", "b4", "b1"]

# B1 is the only experiment that produces a rigged GLB, and in the smoke lane
# it is minutes. Run it first there, so a session you abandon still leaves you
# something to open in Blender.
if SCOPE == "smoke":
    EXPERIMENTS = ["b1"] + [exp for exp in EXPERIMENTS if exp != "b1"]

# Per-experiment ceiling. A run that blows through this is a bug, not progress.
TIMEOUT_MIN = {
    "smoke": {"b1": 45, "b2": 30, "b3": 45, "b4": 120, "b5": 45, "b6": 30, "b7": 30, "b8": 30},
    "full": {"b1": 180, "b2": 120, "b3": 240, "b4": 480, "b5": 240, "b6": 120, "b7": 90, "b8": 120},
}

# Stop LAUNCHING new experiments once the session is this old (minutes).
# Kaggle kills at 12 h (720 min); leave room to collect results.
SESSION_BUDGET_MIN = 660

# Delete each experiment's reconstruction outputs after it finishes. The sparse
# alignment cache is ~1.5 GB per run and accumulates fast.
CLEAN_WORK_AFTER_EACH = True

# Force numpy < 2 to match pyproject. Needs a kernel restart and can break
# preinstalled wheels compiled against numpy 2 -- only set this if the import
# check in the preflight cell actually fails.
PIN_NUMPY_LT2 = False

# Re-run experiments whose CSV already exists.
FORCE_RERUN = False

## 2. Paths and logging

Sets up the log tree and installs handlers that catch tracebacks from notebook
cells, from `warnings`, and from native crashes.

In [ ]:
import datetime as _dt
import faulthandler
import json
import logging
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from collections import deque
from pathlib import Path

faulthandler.enable()


def _detect_platform():
    if Path("/kaggle/working").exists():
        return "kaggle"
    if "google.colab" in sys.modules or Path("/content").exists():
        return "colab"
    return "local"


PLATFORM = _detect_platform()

if PLATFORM == "kaggle":
    # /kaggle/working is persisted but capped (~20 GB) -- results only.
    # /kaggle/temp is ephemeral and roomy -- caches and reconstructions.
    OUT_DIR = Path("/kaggle/working/tier_b")
    WORK_DIR = Path("/kaggle/temp/zshift")
    REPO_DIR = Path("/kaggle/working/z-shift")
elif PLATFORM == "colab":
    drive = Path("/content/drive/MyDrive")
    OUT_DIR = (drive / "zshift-tier-b-results") if drive.exists() else Path("/content/tier_b")
    WORK_DIR = Path("/content/zshift-work")
    REPO_DIR = Path("/content/z-shift")
else:
    OUT_DIR = Path.cwd() / "tier_b_out"
    WORK_DIR = Path.cwd() / "tier_b_work"
    REPO_DIR = Path.cwd()

RESULTS_DIR = OUT_DIR / "results"
RUN_ID = _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
LOG_DIR = OUT_DIR / "logs" / RUN_ID
HF_HOME = WORK_DIR / "hf"

for d in (OUT_DIR, WORK_DIR, RESULTS_DIR, LOG_DIR, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)

SESSION_LOG = LOG_DIR / "session.log"
CRASH_LOG = LOG_DIR / "faulthandler.log"

# Held open for the whole session so a native crash has somewhere to dump.
_crash_fh = open(CRASH_LOG, "w")  # noqa: SIM115
faulthandler.enable(file=_crash_fh, all_threads=True)

log = logging.getLogger("tierb")
log.setLevel(logging.DEBUG)
log.handlers.clear()
log.propagate = False

_fmt = logging.Formatter("%(asctime)s %(levelname)-8s %(name)s: %(message)s", datefmt="%H:%M:%S")
_fh = logging.FileHandler(SESSION_LOG, encoding="utf-8")
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
log.addHandler(_fh)
log.addHandler(_sh)

logging.captureWarnings(True)
_warn_log = logging.getLogger("py.warnings")
_warn_log.handlers.clear()
_warn_log.addHandler(_fh)


def _log_uncaught(exc_type, exc, tb):
    log.error("uncaught exception", exc_info=(exc_type, exc, tb))


sys.excepthook = _log_uncaught

# IPython swallows sys.excepthook, so hook its traceback printer too. Without
# this, a cell that raises leaves nothing in session.log.
try:
    _ip = get_ipython()  # noqa: F821
except NameError:
    _ip = None
if _ip is not None and not getattr(_ip, "_tierb_hooked", False):
    _orig_showtraceback = _ip.showtraceback

    def _showtraceback(*args, **kwargs):
        log.error("cell raised", exc_info=sys.exc_info())
        return _orig_showtraceback(*args, **kwargs)

    _ip.showtraceback = _showtraceback
    _ip._tierb_hooked = True

log.info("platform=%s run_id=%s", PLATFORM, RUN_ID)
log.info("repo=%s", REPO_DIR)
log.info("work=%s (ephemeral: caches, reconstructions)", WORK_DIR)
log.info("out=%s (persisted: CSVs, logs)", OUT_DIR)
log.info("session log -> %s", SESSION_LOG)

## 3. Command runner

Streams output live, tees every line to a log file, caps how much reaches the
notebook (a 3 h run must not bloat the `.ipynb`), and turns a non-zero exit into
an exception carrying the tail of the output.

In [ ]:
import threading

MAX_ECHO_LINES = 400  # per command; the log file always gets everything


def describe_exit(rc):
    if rc == 0:
        return "ok"
    if rc == -9:
        return "SIGKILL -- almost always the OOM killer (host RAM, not VRAM)"
    if rc == -11:
        return "SIGSEGV -- native crash; check faulthandler.log for Python frames"
    if rc == -6:
        return "SIGABRT -- native abort (CUDA / VTK)"
    if rc < 0:
        return "killed by signal " + str(-rc)
    return "non-zero exit " + str(rc)


class CommandFailed(RuntimeError):
    def __init__(self, cmd, returncode, tail, log_path, timed_out=False, timeout_s=None):
        self.cmd = cmd
        self.returncode = returncode
        self.tail = tail
        self.log_path = log_path
        self.timed_out = timed_out
        self.timeout_s = timeout_s
        # A timeout kill also lands as SIGKILL on Linux. Say which one it was,
        # or every timeout gets misread as the OOM killer.
        if timed_out:
            self.reason = "timed out after %.1f min and was killed" % ((timeout_s or 0) / 60)
        else:
            self.reason = describe_exit(returncode)
        super().__init__(f"{self.reason}\n{tail}")


def sh(
    cmd, *, cwd=None, log_path=None, env=None, timeout_s=None, check=True, echo=True, tail_lines=60
):
    # Run cmd (list, or str via bash -lc), tee output to log_path,
    # return (returncode, tail_text).
    if isinstance(cmd, str):
        cmd = ["bash", "-lc", cmd]
    log_path = Path(log_path) if log_path else (LOG_DIR / "commands.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)

    full_env = dict(os.environ)
    if env:
        full_env.update({k: str(v) for k, v in env.items()})

    printable = " ".join(str(c) for c in cmd)
    header = (
        "\n"
        + "=" * 78
        + "\n$ "
        + printable
        + "\n  cwd="
        + str(cwd)
        + "  started="
        + _dt.datetime.now().isoformat(timespec="seconds")
        + "\n"
        + "=" * 78
        + "\n"
    )
    log.debug("running: %s", printable)

    tail = deque(maxlen=max(tail_lines, 200))
    echoed = [0]
    start = time.monotonic()

    with open(log_path, "a", encoding="utf-8", errors="replace") as fh:
        fh.write(header)
        fh.flush()
        proc = subprocess.Popen(  # noqa: S603
            [str(c) for c in cmd],
            cwd=str(cwd) if cwd else None,
            env=full_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            errors="replace",
        )

        def pump():
            for line in proc.stdout:
                fh.write(line)
                tail.append(line)
                if echo:
                    if echoed[0] < MAX_ECHO_LINES:
                        print(line, end="")
                        echoed[0] += 1
                    elif echoed[0] == MAX_ECHO_LINES:
                        print("... output continues in " + str(log_path))
                        echoed[0] += 1
            fh.flush()

        pumper = threading.Thread(target=pump, daemon=True)
        pumper.start()

        timed_out = False
        try:
            rc = proc.wait(timeout=timeout_s)
        except subprocess.TimeoutExpired:
            timed_out = True
            log.error("TIMEOUT after %.1f min, killing: %s", (timeout_s or 0) / 60, cmd[0])
            proc.kill()
            rc = proc.wait()
        pumper.join(timeout=30)
        elapsed = time.monotonic() - start
        fh.write(f"\n--- exit {rc} ({describe_exit(rc)}) after {elapsed:.1f}s ---\n")

    tail_text = "".join(list(tail)[-tail_lines:])
    if timed_out or (check and rc != 0):
        raise CommandFailed(cmd, rc, tail_text, log_path, timed_out=timed_out, timeout_s=timeout_s)
    return rc, tail_text

## 4. Machine report

Recorded so a number in a CSV can never be separated from the box that produced
it. `env_metadata()` already stamps `gpu`, `torch` and `git_commit` onto every
results row; this is the same information, up front, in the log.

In [ ]:
def report_machine():
    log.info("python  %s", platform.python_version())
    log.info("os      %s", platform.platform())
    try:
        import torch

        log.info("torch   %s (cuda %s)", torch.__version__, torch.version.cuda)
        log.info("cuda available: %s", torch.cuda.is_available())
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                log.info(
                    "gpu %d   %s, %.1f GB, cc %d.%d",
                    i,
                    props.name,
                    props.total_memory / 1024**3,
                    props.major,
                    props.minor,
                )
    except Exception:
        log.exception("torch import failed")
    try:
        import numpy

        log.info("numpy   %s", numpy.__version__)
    except Exception:
        log.exception("numpy import failed")

    for label, path in (("work", WORK_DIR), ("out", OUT_DIR)):
        usage = shutil.disk_usage(path)
        log.info(
            "disk %-5s %.1f GB free of %.1f GB  (%s)",
            label,
            usage.free / 1024**3,
            usage.total / 1024**3,
            path,
        )
    try:
        rc, _ = sh("nvidia-smi", log_path=LOG_DIR / "setup.log", check=False)
        if rc != 0:
            log.warning("nvidia-smi returned %s -- is a GPU accelerator selected?", rc)
    except FileNotFoundError:
        log.error("nvidia-smi not found: this runtime has no GPU. Stop and enable one.")


report_machine()

## 5. Clone the repo

No `pip install -e .` of the project. Two reasons:

- `pyproject.toml` pins `requires-python >=3.11,<3.12`, which fails outright on
  a 3.12 image;
- its `torch` / `torchvision` entries resolve from PyPI on Linux (the CUDA index
  is scoped to `sys_platform == 'win32'`), which would replace the preinstalled
  CUDA build with a CPU wheel and quietly cost you the GPU.

Setting `PYTHONPATH` to the repo root plus `src/` imports both `bench` and
`spatial_ingestion` with none of that risk. Nothing in Tier B uses the
`zshift-*` console scripts.

In [ ]:
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    log.info("repo already present, fetching")
    sh(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log")
    sh(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log")
    sh(["git", "pull", "--ff-only"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log", check=False)
else:
    sh(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "50", REPO_URL, str(REPO_DIR)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )

_, _commit = sh(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log", echo=False
)
COMMIT = _commit.strip().splitlines()[-1] if _commit.strip() else "unknown"
log.info("repo at commit %s", COMMIT)

MAST3R_DIR = REPO_DIR / "third_party" / "mast3r"
DUST3R_DIR = MAST3R_DIR / "dust3r"

# Repo root for `bench`, src/ for `spatial_ingestion`.
PYTHONPATH = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "src")])

CHILD_ENV = {
    "PYTHONPATH": PYTHONPATH,
    "PYTHONUNBUFFERED": "1",
    "PYTHONFAULTHANDLER": "1",
    "HF_HOME": str(HF_HOME),
    "HUGGINGFACE_HUB_CACHE": str(HF_HOME / "hub"),
    "TOKENIZERS_PARALLELISM": "false",
    # PyVista/VTK are used for mesh filtering, never rendering, but this
    # runtime is headless -- make the intent explicit.
    "PYVISTA_OFF_SCREEN": "true",
    "MPLBACKEND": "Agg",
}

missing = [p for p in ("bench", "src/spatial_ingestion") if not (REPO_DIR / p).exists()]
if missing:
    raise SystemExit(
        f"missing from the clone: {missing}. These are untracked on the laptop -- "
        "commit and push them, then re-run."
    )
log.info("bench/ and src/spatial_ingestion/ present")

## 6. Dependencies

Read straight out of `pyproject.toml`, minus `torch`/`torchvision` (keep the
preinstalled CUDA build) and minus `numpy` unless you opted into the `<2` pin.

In [ ]:
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib  # type: ignore

with open(REPO_DIR / "pyproject.toml", "rb") as fh:
    _pyproject = tomllib.load(fh)

SKIP_PREFIXES = ("torch", "torchvision")
deps = []
for dep in _pyproject["project"]["dependencies"]:
    name = dep.split(">")[0].split("<")[0].split("=")[0].split("[")[0].strip().lower()
    if name.startswith(SKIP_PREFIXES):
        log.info("skipping %-12s (keeping the preinstalled CUDA build)", name)
        continue
    if name == "numpy" and not PIN_NUMPY_LT2:
        log.info("skipping %-12s (set PIN_NUMPY_LT2 only if imports actually fail)", name)
        continue
    deps.append(dep)

log.info("installing %d dependencies", len(deps))
sh(
    [sys.executable, "-m", "pip", "install", "-q", *deps],
    log_path=LOG_DIR / "setup.log",
    timeout_s=1800,
)
log.info("dependencies installed")
if PIN_NUMPY_LT2:
    log.warning("numpy pinned <2 -- RESTART THE KERNEL now, then re-run from cell 2")

## 7. MASt3R

Mirrors `scripts/setup-mast3r.sh` (same pinned commit) but installs with
`--no-deps`: MASt3R's and DUSt3R's own `requirements.txt` list `torch`, and
letting pip resolve them is the other way to lose the CUDA build. Their real
dependencies are already in the project's list, which is why `roma`, `einops`,
`pyglet<2` and friends are there.

The RoPE CUDA kernel compile fails on your laptop and should succeed here. That
is a genuinely different code path from the one Tier A measured, and belongs in
the paper's setup section.

In [ ]:
PINNED_MAST3R = "f5209afc300cec36239a7ac992263f36847bbba0"

PY_STUB = """[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "{name}"
version = "0.1.0"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["."]
include = ["{name}*"]
"""

if not MAST3R_DIR.exists():
    sh(
        ["git", "clone", "https://github.com/naver/mast3r", str(MAST3R_DIR)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
    sh(["git", "checkout", PINNED_MAST3R], cwd=MAST3R_DIR, log_path=LOG_DIR / "setup.log")
    sh(
        ["git", "submodule", "update", "--init", "--recursive"],
        cwd=MAST3R_DIR,
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
else:
    log.info("mast3r already cloned at %s", MAST3R_DIR)

for target, name in ((MAST3R_DIR, "mast3r"), (DUST3R_DIR, "dust3r")):
    stub = target / "pyproject.toml"
    if not stub.exists():
        stub.write_text(PY_STUB.format(name=name), encoding="utf-8")
        log.info("wrote %s", stub)
    sh(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(target)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=900,
    )

# RoPE CUDA kernels: a speedup, not a requirement. Never fatal.
curope = DUST3R_DIR / "croco" / "models" / "curope"
try:
    sh(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=curope,
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
    log.info("RoPE CUDA kernels compiled")
    CUROPE_BUILT = True
except CommandFailed as exc:
    CUROPE_BUILT = False
    log.warning(
        "RoPE kernel compile failed (%s); falling back to the PyTorch path. "
        "Not fatal, but note it in the setup section. Log: %s",
        describe_exit(exc.returncode),
        exc.log_path,
    )

## 8. Kaggle datasets

Kaggle mounts every attached dataset read-only at `/kaggle/input/<slug>`, one
directory per dataset, with no re-download between sessions. This cell decides
which scenes Tier B runs on:

1. resolve `SCENES` against `DATA_ROOT`;
2. if nothing resolves and `AUTO_DISCOVER_SCENES` is on, walk `DATA_ROOT` and
   then every mount, and build the scene list from what is actually there.

**No dataset slug is hardcoded.** Redistributions of DTU and BlendedMVS appear
and disappear from Kaggle, so the notebook reads the mount rather than a list of
names. Attach whatever you have; these layouts are recognised:

| Layout | Looks like | Scene name | Ground truth matched by |
|---|---|---|---|
| DTU | `Rectified/scan24/*.png` + `Points/stl/stl024_total.ply` | `scan24` | scan number, anywhere on the mount |
| BlendedMVS | `<id>/blended_images/*.jpg` | `<id>` | a cloud beside the images |
| Tanks and Temples, COLMAP | `<scene>/images/*.jpg` + `<scene>/*.ply` | `<scene>` | one level up from the images |
| own capture | `captures/desk_orbit/frames/*.jpg` | `desk_orbit` | a cloud beside the images |
| anything else | a folder holding >= 4 images | the folder | beside, or one level up |

`EXTRA_KAGGLE_DATASETS` pulls a slug with `kagglehub` for a dataset you did not
attach. That needs *Settings > Internet* on and writes to the working disk, so
attaching is still the better option.

A mount with no `.ply` gives you **no ground truth**, and B2/B4/B6/B8 skip
scenes with `gt_path=None` -- they exit 0 having written zero rows. This cell
says so loudly rather than letting you find out three hours in.

In [ ]:
import re

# Kaggle mounts every attached dataset read-only at /kaggle/input/<slug>, one
# directory per dataset. Discovery reads the mount instead of a hardcoded list
# of dataset names: redistributions of DTU and BlendedMVS appear and disappear
# from Kaggle, so a manifest pinned to one particular slug rots.

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
GT_SUFFIXES = {".ply", ".obj", ".off", ".stl", ".xyz", ".pcd"}

# A directory holding at least this many images is a candidate scene.
MIN_SCENE_IMAGES = 4

# Directory names that mean "the images live here" -- such a scene is named
# after its parent instead. Covers DTU (Rectified/scan24/), BlendedMVS
# (<id>/blended_images/), COLMAP and Tanks and Temples (<scene>/images/) and
# our own extracted captures (<scene>/frames/).
IMAGE_DIR_NAMES = {
    "images",
    "image",
    "rect",
    "rectified",
    "blended_images",
    "frames",
    "rgb",
    "color",
    "dense",
    "train",
    "test",
    "val",
}

# Walking a 50 GB mount must not become the slow part of the notebook.
MAX_WALK_DIRS = 20000
MAX_GT_FILES = 5000

KAGGLE_INPUT = Path("/kaggle/input")


def _images_in(directory):
    try:
        return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)
    except OSError:
        return []


def _first_number(text):
    match = re.search(r"(\d+)", str(text))
    return int(match.group(1)) if match else None


def scan_tree(root):
    # One walk per mount: every candidate image dir, and every GT-shaped file.
    image_dirs, gt_files, truncated = [], [], False
    for n_dirs, (dirpath, dirnames, filenames) in enumerate(os.walk(root), start=1):
        dirnames[:] = sorted(d for d in dirnames if not d.startswith("."))
        if n_dirs > MAX_WALK_DIRS:
            truncated = True
            break
        here = Path(dirpath)
        n_images = sum(1 for f in filenames if Path(f).suffix.lower() in IMAGE_SUFFIXES)
        if n_images >= MIN_SCENE_IMAGES:
            image_dirs.append((here, n_images))
        if len(gt_files) < MAX_GT_FILES:
            gt_files.extend(here / f for f in filenames if Path(f).suffix.lower() in GT_SUFFIXES)
    return image_dirs, gt_files, truncated


def _match_gt(scene_name, image_dir, gt_files):
    # Beside the images first, then one level up (COLMAP, Tanks and Temples),
    # then by scan number anywhere on the mount -- which is what DTU needs:
    # Rectified/scan24/ and Points/stl/stl024_total.ply share nothing but "24".
    beside = [g for g in gt_files if g.parent in (image_dir, image_dir.parent)]
    if beside:
        return max(beside, key=lambda p: p.stat().st_size)
    number = _first_number(scene_name)
    if number is None:
        return None
    numbered = [g for g in gt_files if _first_number(g.stem) == number]
    return max(numbered, key=lambda p: p.stat().st_size) if numbered else None


def discover_scenes(root, limit=None, used_names=None):
    limit = MAX_DISCOVERED_SCENES if limit is None else limit
    used_names = set() if used_names is None else used_names
    root = Path(root)
    if not root.is_dir() or limit <= 0:
        return []
    image_dirs, gt_files, truncated = scan_tree(root)
    if truncated:
        log.warning(
            "stopped walking %s after %d directories -- discovery is partial; "
            "point DATA_ROOT at the subtree you care about",
            root,
            MAX_WALK_DIRS,
        )

    found = []
    for image_dir, n_images in sorted(image_dirs, key=lambda item: (-item[1], str(item[0]))):
        label = image_dir.name
        if label.lower() in IMAGE_DIR_NAMES and image_dir.parent != root:
            label = image_dir.parent.name
        name, suffix = label, 2
        while name in used_names:
            name = f"{label}_{suffix}"
            suffix += 1
        used_names.add(name)
        gt = _match_gt(label, image_dir, gt_files)
        found.append(
            {
                "name": name,
                "image_dir": str(image_dir),
                "gt_path": str(gt) if gt else None,
                "tau": DEFAULT_TAU,
                "units": DEFAULT_UNITS,
                "notes": f"auto-discovered under {root}",
                "n_images": n_images,
            }
        )
        if len(found) >= limit:
            break
    return found


def resolve_configured(scenes, data_root):
    # SCENES holds paths relative to DATA_ROOT. Resolve them to absolute paths
    # and say which ones are absent here, rather than failing later.
    data_root = Path(data_root)
    resolved, missing = [], []
    for scene in scenes:
        image_dir = (data_root / scene["image_dir"]).resolve()
        if not image_dir.is_dir():
            missing.append(f"{scene['name']}: {image_dir}")
            continue
        gt = scene.get("gt_path")
        gt_path = (data_root / gt).resolve() if gt else None
        if gt_path is not None and not gt_path.exists():
            log.warning("%s: gt_path does not exist: %s", scene["name"], gt_path)
            gt_path = None
        entry = dict(scene)
        entry["image_dir"] = str(image_dir)
        entry["gt_path"] = str(gt_path) if gt_path else None
        entry["n_images"] = len(_images_in(image_dir))
        resolved.append(entry)
    return resolved, missing


FETCHED_ROOTS = []


def fetch_kaggle_datasets(slugs):
    # Optional. Attaching a dataset in the sidebar is better: it mounts
    # instantly and needs no internet. This is for a slug you did not attach,
    # and for Colab, which has no /kaggle/input at all.
    downloaded = []
    for slug in slugs:
        try:
            import kagglehub
        except ModuleNotFoundError:
            sh(
                [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
                log_path=LOG_DIR / "setup.log",
                timeout_s=600,
            )
            import kagglehub
        try:
            path = Path(kagglehub.dataset_download(slug))
        except Exception:
            log.exception(
                "could not download %s -- attach it as a dataset instead, "
                "or turn on Settings > Internet",
                slug,
            )
            continue
        log.info("downloaded %-40s -> %s", slug, path)
        downloaded.append(path)
        FETCHED_ROOTS.append(path)
    return downloaded


def _mount_for(slug):
    # Kaggle mounts a dataset under the second half of its slug.
    name = slug.split("/")[-1]
    direct = KAGGLE_INPUT / name
    if direct.is_dir():
        return direct
    # A renamed or hand-uploaded copy: match on the slug's distinctive words.
    words = [word for word in re.split(r"[-_]", name) if len(word) > 3][:2]
    for mount in MOUNTS:
        if words and all(word in mount.name.lower() for word in words):
            return mount
    return None


def resolve_kaggle_scenes(entries):
    # Resolve each configured Kaggle dataset to a scene on this machine. The
    # image_dir hint is trusted when it lands on images; otherwise the mount is
    # walked, so a mirror that repackaged the tree still resolves rather than
    # taking down the whole notebook.
    resolved, missing = [], []
    for entry in entries:
        slug = entry["slug"]
        mount = _mount_for(slug)
        if mount is None and FETCH_MISSING_DATASETS:
            fetched = fetch_kaggle_datasets([slug])
            mount = fetched[0] if fetched else None
        if mount is None:
            missing.append(f"{entry['name']}: {slug} is not attached (Input > Add Input)")
            continue

        hinted = mount / (entry.get("image_dir") or "")
        image_dir = hinted if _images_in(hinted) else None
        gt_hint = entry.get("gt_path")
        gt_path = (mount / gt_hint) if gt_hint else None

        if image_dir is None:
            log.warning("%s: no images at %s -- walking %s instead", entry["name"], hinted, mount)
            found = discover_scenes(mount, limit=1)
            if not found:
                missing.append(f"{entry['name']}: no images anywhere under {mount}")
                continue
            image_dir = Path(found[0]["image_dir"])
            if gt_path is None and found[0]["gt_path"]:
                gt_path = Path(found[0]["gt_path"])

        if gt_path is not None and not gt_path.exists():
            log.warning("%s: gt_path does not exist: %s", entry["name"], gt_path)
            gt_path = None

        if gt_path is None:
            # Nothing configured: look for a cloud on this mount anyway. A
            # COLMAP points3D.ply is pseudo-GT, not sensor ground truth -- say
            # so in the paper. Section 10 checks tau against its bounding
            # diagonal, which is what catches a cloud in the wrong units.
            _, gt_files, _ = scan_tree(mount)
            candidate = _match_gt(entry["name"], image_dir, gt_files)
            if candidate is None and len(gt_files) == 1:
                candidate = gt_files[0]
            if candidate is not None:
                log.warning(
                    "%s: no gt_path configured; taking %s as pseudo-GT. Verify it "
                    "before reporting any F-score against it.",
                    entry["name"],
                    candidate,
                )
                gt_path = candidate

        resolved.append(
            {
                "name": entry["name"],
                "image_dir": str(image_dir),
                "gt_path": str(gt_path) if gt_path else None,
                "tau": entry.get("tau", DEFAULT_TAU),
                "units": entry.get("units", DEFAULT_UNITS),
                "notes": f"{entry.get('notes', '')} [kaggle: {slug}]".strip(),
                "n_images": len(_images_in(image_dir)),
            }
        )
    return resolved, missing


# --- what is mounted ------------------------------------------------------
if KAGGLE_INPUT.is_dir():
    MOUNTS = sorted(p for p in KAGGLE_INPUT.iterdir() if p.is_dir())
    log.info("%d dataset(s) mounted at %s", len(MOUNTS), KAGGLE_INPUT)
    for mount in MOUNTS:
        n_files = sum(1 for p in mount.rglob("*") if p.is_file())
        log.info("  %-44s %6d files", mount.name, n_files)
    if not MOUNTS:
        log.warning("nothing attached: Kaggle sidebar > Input > Add Input")
else:
    MOUNTS = []
    log.info("no %s (not on Kaggle); using DATA_ROOT=%s", KAGGLE_INPUT, DATA_ROOT)

# --- resolve the scene list -----------------------------------------------
RESOLVED_SCENES, MISSING_SCENES = resolve_configured(SCENES, DATA_ROOT)
_kaggle_scenes, _kaggle_missing = resolve_kaggle_scenes(KAGGLE_SCENES)
RESOLVED_SCENES.extend(_kaggle_scenes)
MISSING_SCENES.extend(_kaggle_missing)
for miss in MISSING_SCENES:
    log.warning("scene not resolved: %s", miss)

if not RESOLVED_SCENES and AUTO_DISCOVER_SCENES:
    # Every root, not just the first that yields something: a session with the
    # capture dataset and DTU both attached must not silently drop the one that
    # carries ground truth.
    seen_roots, seen_dirs, used_names = set(), set(), set()
    for search_root in (Path(DATA_ROOT), *FETCHED_ROOTS, *MOUNTS):
        if search_root in seen_roots or not search_root.is_dir():
            continue
        seen_roots.add(search_root)
        log.info("no configured scene resolved; walking %s", search_root)
        for scene in discover_scenes(
            search_root, MAX_DISCOVERED_SCENES - len(RESOLVED_SCENES), used_names
        ):
            if scene["image_dir"] in seen_dirs:
                continue
            seen_dirs.add(scene["image_dir"])
            RESOLVED_SCENES.append(scene)
        if len(RESOLVED_SCENES) >= MAX_DISCOVERED_SCENES:
            log.warning(
                "stopped at MAX_DISCOVERED_SCENES=%d; raise it to take in more",
                MAX_DISCOVERED_SCENES,
            )
            break

if not RESOLVED_SCENES:
    raise SystemExit(
        "no scenes. Attach a dataset (Kaggle sidebar > Input > Add Input), point "
        "DATA_ROOT at its mount and list the scenes in SCENES -- or leave "
        "AUTO_DISCOVER_SCENES on and let this cell find them."
    )

print(f"{'scene':<24} {'images':>7}  {'gt':<5}  image_dir")
print("-" * 96)
for scene in RESOLVED_SCENES:
    print(
        f"{scene['name']:<24} {scene.get('n_images', 0):>7}  "
        f"{'yes' if scene.get('gt_path') else 'NO':<5}  {scene['image_dir']}"
    )
N_WITH_GT = sum(1 for s in RESOLVED_SCENES if s.get("gt_path"))
print(f"\n{len(RESOLVED_SCENES)} scene(s), {N_WITH_GT} with ground truth")
if N_WITH_GT == 0:
    log.warning(
        "no scene has ground truth. B2/B4/B6/B8 skip scenes with gt_path=None "
        "and will produce ZERO ROWS -- only B1/B3/B5/B7 mean anything here."
    )

## 9. Scene manifest

Written into the work dir with **absolute** paths, so a read-only
`/kaggle/input` mount works and nothing has to be copied. Only the manifest
keys are carried over: discovery and the dataset test hang working state off
the same dicts.

In [ ]:
MANIFEST_PATH = WORK_DIR / "scenes.json"

# Discovery and the dataset test hang working state (frame counts, loaded
# paths) off the same scene dicts, so the manifest takes named keys only.
MANIFEST_KEYS = ("name", "image_dir", "gt_path", "tau", "units", "notes")

manifest = {
    "_generated_by": "tier_b_gpu.ipynb run " + RUN_ID,
    "tau": DEFAULT_TAU,
    "units": DEFAULT_UNITS,
    "scenes": [
        {key: scene[key] for key in MANIFEST_KEYS if key in scene} for scene in RESOLVED_SCENES
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
log.info("manifest -> %s (%d scenes)", MANIFEST_PATH, len(manifest["scenes"]))
print(MANIFEST_PATH.read_text())

## 10. Dataset test

A pytest-shaped pass over the mounted data, in this kernel, in seconds. It runs
*before* the checkpoint download and the GPU work because every failure below is
cheap here and expensive later.

| Check | Fails when | Why it is here |
|---|---|---|
| `images` | fewer than 2 images | MASt3R needs a pair. Warns at <= 40 frames, where B4's budget never fires and its three variants come out identical |
| `decode` | any file fails its header check, or a sampled full decode comes back `None` | every image is header-checked and up to 8 are fully decoded; one corrupt frame takes a run down in the middle of the grid. Warns on mixed resolutions inside one scene |
| `capture_order` | *(warn only)* | `Scene.image_paths` sorts lexically and every B module reads that as capture order; unpadded numbering silently reverses it (`frame10` before `frame2`) -- the defect A5 measured |
| `ground_truth` | trimesh cannot load it, fewer than 1000 points, or non-finite coordinates | loaded through `bench.tier_b_common.load_gt_points`, so a pass here is a pass in B2 |
| `tau_vs_gt_scale` | *(warn only)* | `tau` outside 0.01%-10% of the GT bounding diagonal means the manifest and the GT file disagree about units. A metres-vs-mm mix-up yields a perfectly plausible F-score that means nothing |
| `exif_intrinsics` | *(warn only)* | B8 compares EXIF-derived intrinsics against the default guess; no EXIF means B8 measures nothing on that scene |
| `input_mount` | nothing is attached | catches a notebook pointed at a dataset that was never added |
| `manifest_roundtrip` | `SceneSet.from_manifest` changes the scene list, or `image_paths` comes back unsorted | reads the manifest just written back through the exact class B1-B8 use |

Statuses are `ok` / `WARN` (read it, keep going) / `skip` (could not run) /
`FAIL`. Failures stop the notebook while `DATASET_TEST_STRICT` is on. The table
truncates each detail; the full record goes to
`logs/<run_id>/dataset_test.json`.

In [ ]:
import functools

import numpy as np

# Import the repo into this kernel so the checks use exactly the loaders Tier B
# uses: a ground-truth file trimesh cannot open here will not open in B2 either.
for _path in (str(REPO_DIR), str(REPO_DIR / "src")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

try:
    from bench.tier_b_common import SceneSet, load_gt_points
    from spatial_ingestion.config import MAX_RECONSTRUCTION_FRAMES
except Exception as exc:
    SceneSet, load_gt_points, MAX_RECONSTRUCTION_FRAMES = None, None, 40
    log.warning("bench not importable in this kernel (%r); GT checks will be skipped", exc)


class CheckSkipped(Exception):
    """The check could not run: no ground truth, no EXIF, no bench import."""


class CheckWarning(Exception):
    """It ran and found something you should read, but not a reason to stop."""


# Header-check every image; fully decode at most this many, evenly spaced.
MAX_FULL_DECODES = 8

RESULTS = []


def check_case(scene_name, check_name, fn):
    try:
        detail, status = fn(), "pass"
    except CheckSkipped as exc:
        detail, status = str(exc), "skip"
    except CheckWarning as exc:
        detail, status = str(exc), "warn"
    except Exception as exc:
        detail, status = f"{type(exc).__name__}: {exc}", "fail"
        log.exception("FAIL  %s / %s", scene_name, check_name)
    RESULTS.append(
        {"scene": scene_name, "check": check_name, "status": status, "detail": str(detail)}
    )
    return status


# --- per-scene checks -----------------------------------------------------
def _check_images(scene):
    paths = _images_in(scene["image_dir"])
    scene["_paths"] = paths
    if len(paths) < 2:
        raise RuntimeError(f"{len(paths)} image(s) -- MASt3R needs a pair at minimum")
    if len(paths) <= MAX_RECONSTRUCTION_FRAMES:
        raise CheckWarning(
            f"{len(paths)} frames <= MAX_RECONSTRUCTION_FRAMES={MAX_RECONSTRUCTION_FRAMES}: "
            "the budget never fires, so B4's three variants come out identical"
        )
    return f"{len(paths)} images"


def _check_decode(scene):
    # Every file gets a header check: one corrupt frame takes a run down in the
    # middle of the grid, and catching it here costs a second. A spaced sample
    # is fully decoded on top -- that is what catches truncation past the
    # header, and it is where the resolution comes from.
    import cv2
    from PIL import Image

    paths = scene.get("_paths") or []
    if not paths:
        raise CheckSkipped("no images")

    unreadable = []
    for path in paths:
        try:
            with Image.open(path) as image:
                image.verify()
        except Exception as exc:
            unreadable.append(f"{path.name} ({type(exc).__name__})")
    if unreadable:
        raise RuntimeError(f"{len(unreadable)} unreadable file(s): " + ", ".join(unreadable[:4]))

    sample = paths[:: max(1, len(paths) // MAX_FULL_DECODES)][:MAX_FULL_DECODES]
    shapes = []
    for path in sample:
        image = cv2.imread(str(path))
        if image is None:
            raise RuntimeError(f"cv2 returned None for {path.name} -- truncated, or an odd depth")
        shapes.append(image.shape[:2])
    scene["_resolution"] = shapes[0]
    if len(set(shapes)) > 1:
        raise CheckWarning(
            "mixed resolutions inside one scene: "
            + ", ".join(f"{w}x{h}" for h, w in sorted(set(shapes)))
        )
    height, width = shapes[0]
    return f"{width}x{height}, {len(paths)} headers + {len(sample)} full decodes"


def _check_order(scene):
    # Scene.image_paths sorts lexically and every B module treats that as
    # capture order. Unpadded numbering breaks it silently (frame10 sorts before
    # frame2) -- the same ordering defect A5 measured on the CPU side.
    #
    # Compare every numeric field, not just one: DTU's rect_001_0_r5000.png
    # carries the frame index first and a lighting index last, so reading a
    # single field passes vacuously on the wrong number.
    paths = scene.get("_paths") or []
    if not paths:
        raise CheckSkipped("no images")
    keys = [tuple(int(number) for number in re.findall(r"\d+", path.stem)) for path in paths]
    if not all(keys):
        raise CheckWarning("some filenames carry no number; capture order is the filesystem's")
    if len({len(key) for key in keys}) > 1:
        raise CheckWarning("filenames do not share one numbering scheme; check the capture order")
    if keys != sorted(keys):
        first_bad = next(i for i in range(1, len(keys)) if keys[i] < keys[i - 1])
        raise CheckWarning(
            f"lexical order is not numeric order ({paths[first_bad - 1].name} sorts before "
            f"{paths[first_bad].name}): zero-pad the numbers, or B4 reconstructs out of order"
        )
    return f"{len(paths)} names in numeric order, {len(keys[0])} numeric field(s)"


def _check_gt(scene):
    gt_path = scene.get("gt_path")
    if not gt_path:
        raise CheckSkipped("gt_path is null -- B2/B4/B6/B8 skip this scene")
    if load_gt_points is None:
        raise CheckSkipped("bench.tier_b_common not importable in this kernel")
    points = load_gt_points(Path(gt_path), max_points=50000)
    if len(points) < 1000:
        raise RuntimeError(f"only {len(points)} ground-truth points")
    if not np.isfinite(points).all():
        raise RuntimeError("ground truth contains non-finite coordinates")
    extent = points.max(axis=0) - points.min(axis=0)
    scene["_gt_diagonal"] = float(np.linalg.norm(extent))
    return (
        f"{len(points)} pts, bbox {extent[0]:.3g} x {extent[1]:.3g} x {extent[2]:.3g} "
        f"{scene.get('units', '?')}"
    )


def _check_tau(scene):
    # tau stated in the wrong unit is the failure that produces a plausible
    # F-score meaning nothing. DTU's 2 mm against a ~300 mm object is 0.7% of
    # the bounding diagonal; outside 0.01%-10% the manifest and the GT file
    # almost certainly disagree about units.
    diagonal = scene.get("_gt_diagonal")
    if diagonal is None:
        raise CheckSkipped("ground truth not loaded")
    tau = float(scene.get("tau", DEFAULT_TAU))
    ratio = tau / diagonal
    if not 1e-4 <= ratio <= 1e-1:
        raise CheckWarning(
            f"tau={tau} {scene.get('units', '?')} is {ratio:.3%} of the GT bounding diagonal "
            f"({diagonal:.4g}) -- check the units before trusting any F-score"
        )
    return f"tau is {ratio:.3%} of the GT diagonal"


def _check_exif(scene):
    # B8 compares EXIF-derived intrinsics against the default guess. DTU carries
    # no EXIF, a phone capture does; either is fine, but B8 measures nothing on
    # a scene without it.
    paths = scene.get("_paths") or []
    if not paths:
        raise CheckSkipped("no images")
    from PIL import ExifTags, Image

    with Image.open(paths[0]) as image:
        raw = dict(image.getexif())
    tags = {ExifTags.TAGS.get(key, key): value for key, value in raw.items()}
    wanted = ("FocalLength", "FocalLengthIn35mmFilm", "Make", "Model")
    present = {key: tags[key] for key in wanted if key in tags}
    if not {"FocalLength", "FocalLengthIn35mmFilm"} & set(present):
        raise CheckWarning("no EXIF focal length: B8 has nothing to compare against on this scene")
    return ", ".join(f"{key}={value}" for key, value in present.items())


# --- dataset-wide checks --------------------------------------------------
def _check_mount():
    if not KAGGLE_INPUT.is_dir():
        raise CheckSkipped(f"{KAGGLE_INPUT} does not exist -- not running on Kaggle")
    if not MOUNTS:
        raise RuntimeError("no dataset attached: Kaggle sidebar > Input > Add Input")
    used = [
        mount
        for mount in MOUNTS
        if any(scene["image_dir"].startswith(str(mount)) for scene in RESOLVED_SCENES)
    ]
    if not used:
        raise CheckWarning(f"{len(MOUNTS)} dataset(s) mounted, but no scene comes from any of them")
    return f"{len(used)} of {len(MOUNTS)} mount(s) in use: " + ", ".join(m.name for m in used)


def _check_manifest():
    if SceneSet is None:
        raise CheckSkipped("bench.tier_b_common not importable in this kernel")
    scene_set = SceneSet.from_manifest(MANIFEST_PATH)
    names = [scene.name for scene in scene_set.scenes]
    if names != [scene["name"] for scene in RESOLVED_SCENES]:
        raise RuntimeError(f"round-trip changed the scene list: {names}")
    for scene in scene_set.scenes:
        wanted = min(4, len(_images_in(scene.image_dir)))
        paths = scene.image_paths(limit=wanted)
        if len(paths) != wanted:
            raise RuntimeError(f"{scene.name}: image_paths(limit={wanted}) returned {len(paths)}")
        if paths != sorted(paths):
            raise RuntimeError(f"{scene.name}: image_paths did not come back sorted")
    return f"{len(names)} scene(s) survived SceneSet.from_manifest"


SCENE_CHECKS = (
    ("images", _check_images),
    ("decode", _check_decode),
    ("capture_order", _check_order),
    ("ground_truth", _check_gt),
    ("tau_vs_gt_scale", _check_tau),
    ("exif_intrinsics", _check_exif),
)

log.info("dataset test: %d scene(s) x %d checks", len(RESOLVED_SCENES), len(SCENE_CHECKS))
for scene in RESOLVED_SCENES:
    for check_name, check_fn in SCENE_CHECKS:
        check_case(scene["name"], check_name, functools.partial(check_fn, scene))
check_case("-", "input_mount", _check_mount)
check_case("-", "manifest_roundtrip", _check_manifest)

MARK = {"pass": "ok", "warn": "WARN", "fail": "FAIL", "skip": "skip"}
print()
print("=" * 100)
print(f"{'scene':<18} {'check':<18} {'':<4} detail")
print("-" * 100)
for row in RESULTS:
    print(f"{row['scene']:<18} {row['check']:<18} {MARK[row['status']]:<4} {row['detail'][:56]}")
print("=" * 100)

TALLY = {status: sum(1 for row in RESULTS if row["status"] == status) for status in MARK}
print(
    f"{TALLY['pass']} passed, {TALLY['warn']} warned, {TALLY['skip']} skipped, "
    f"{TALLY['fail']} failed"
)

DATASET_TEST_PATH = LOG_DIR / "dataset_test.json"
DATASET_TEST_PATH.write_text(
    json.dumps(
        {
            "run_id": RUN_ID,
            "data_root": str(DATA_ROOT),
            "manifest": str(MANIFEST_PATH),
            "tally": TALLY,
            "results": RESULTS,
        },
        indent=2,
    ),
    encoding="utf-8",
)
log.info("dataset test -> %s (full detail; the table above is truncated)", DATASET_TEST_PATH)

if TALLY["fail"] and DATASET_TEST_STRICT:
    raise SystemExit(
        f"{TALLY['fail']} dataset check(s) failed. Fix the data before spending GPU hours "
        f"on it, or set DATASET_TEST_STRICT = False to proceed anyway. "
        f"Detail: {DATASET_TEST_PATH}"
    )

## 11. Preflight

A hard gate. Everything that can fail cheaply fails here, before hours of
compute: imports, CUDA, checkpoint download, every scene path, every GT file,
frame counts, disk. Fatal problems raise; the rest are warnings you should read.

In [ ]:
FATAL = []
WARN = []


def check(label, fn, fatal=True):
    try:
        result = fn()
    except Exception as exc:
        (FATAL if fatal else WARN).append(f"{label}: {exc!r}")
        log.exception("FAIL  %s", label)
        return None
    log.info("ok    %-34s %s", label, "" if result is None else result)
    return result


def _imports():
    mods = (
        "torch",
        "numpy",
        "cv2",
        "trimesh",
        "pyvista",
        "scipy",
        "sklearn",
        "roma",
        "einops",
        "mast3r.model",
        "dust3r.utils.image",
        "spatial_ingestion.reconstruction.pipeline",
        "spatial_ingestion.final_pipeline.handoff",
        "bench.tier_b_common",
    )
    src = (
        "import importlib\n"
        + "\n".join(f"importlib.import_module({m!r})" for m in mods)
        + "\nprint('imports ok')"
    )
    sh(
        [sys.executable, "-c", src],
        cwd=REPO_DIR,
        env=CHILD_ENV,
        log_path=LOG_DIR / "preflight.log",
        echo=False,
        timeout_s=600,
    )
    return f"{len(mods)} modules"


def _cuda():
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "torch.cuda.is_available() is False. Enable a GPU accelerator "
            "(Kaggle: Settings > Accelerator; Colab: Runtime > Change runtime type)."
        )
    free, total = torch.cuda.mem_get_info(0)
    name = torch.cuda.get_device_name(0)
    return f"{name}, {free / 1024**3:.1f}/{total / 1024**3:.1f} GB free"


def _checkpoint():
    # Download the weights here, where a network failure costs seconds.
    src = (
        "import time\n"
        "from mast3r.model import AsymmetricMASt3R\n"
        "t = time.time()\n"
        "m = AsymmetricMASt3R.from_pretrained("
        "'naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric')\n"
        "m = m.to('cuda').eval()\n"
        "n = sum(p.numel() for p in m.parameters())\n"
        "print('loaded %.0fM params to cuda in %.1fs' % (n / 1e6, time.time() - t))\n"
    )
    _, tail = sh(
        [sys.executable, "-c", src],
        cwd=REPO_DIR,
        env=CHILD_ENV,
        log_path=LOG_DIR / "preflight.log",
        timeout_s=1800,
        echo=False,
    )
    lines = [ln for ln in tail.strip().splitlines() if ln.strip()]
    return lines[-1] if lines else "loaded"


def _scenes():
    entries = json.loads(MANIFEST_PATH.read_text())["scenes"]
    if not entries:
        raise RuntimeError("manifest has no scenes")
    suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    summary = []
    for entry in entries:
        image_dir = Path(entry["image_dir"])
        if not image_dir.is_dir():
            raise FileNotFoundError(f"{entry['name']}: image_dir does not exist: {image_dir}")
        n = sum(1 for p in image_dir.iterdir() if p.suffix.lower() in suffixes)
        if n == 0:
            raise FileNotFoundError(f"{entry['name']}: no images under {image_dir}")
        gt = entry.get("gt_path")
        if not gt:
            WARN.append(
                f"{entry['name']}: gt_path is null. B2/B4/B6/B8 skip scenes with no GT -- "
                "this scene contributes zero rows."
            )
        elif not Path(gt).exists():
            raise FileNotFoundError(f"{entry['name']}: gt_path missing: {gt}")
        if n <= 40:
            WARN.append(
                f"{entry['name']}: only {n} frames. B4 caps at MAX_RECONSTRUCTION_FRAMES=40, "
                "so the budget never fires and its three variants are identical."
            )
        summary.append(f"{entry['name']}={n} frames")
    return ", ".join(summary)


def _disk():
    problems = []
    for label, path, need_gb in (("work", WORK_DIR, 25), ("out", OUT_DIR, 2)):
        free = shutil.disk_usage(path).free / 1024**3
        if free < need_gb:
            problems.append(f"{label} has {free:.1f} GB free, want >= {need_gb} GB")
    if problems:
        raise RuntimeError("; ".join(problems))
    return "sufficient"


check("imports", _imports)
check("cuda", _cuda)
check("scenes", _scenes)
check("disk", _disk)
check("mast3r checkpoint", _checkpoint)

print()
for w in WARN:
    log.warning("WARN  %s", w)
if FATAL:
    for f in FATAL:
        log.error("FATAL %s", f)
    raise SystemExit(
        f"{len(FATAL)} preflight failure(s). Fix these before running anything long. "
        f"Full detail: {LOG_DIR / 'preflight.log'}"
    )
log.info("preflight passed (%d warnings)", len(WARN))

## 12. Experiment table

`--quick` is deliberately never passed: no `exp_b*.py` module forwards it to
`run()`, so it would do nothing while looking like it did something. Scope comes
from the flags each module actually reads.

Note B4: its frame budget is the module constant `DEFAULT_BUDGET = 40` with no
CLI override, so even the smoke lane runs three 40-frame reconstructions. It is
the longest smoke item by a wide margin, which is why it sits late in
`EXPERIMENTS`. Wiring `--quick` into `run()` is the real fix.

In [ ]:
MODULES = {
    "b1": ("b1_end_to_end", "bench.exp_b1_end_to_end"),
    "b2": ("b2_reconstruction_accuracy", "bench.exp_b2_reconstruction_accuracy"),
    "b3": ("b3_pairing_ablation", "bench.exp_b3_pairing_ablation"),
    "b4": ("b4_frame_budget_ablation", "bench.exp_b4_frame_budget_ablation"),
    "b5": ("b5_tsdf_fallback", "bench.exp_b5_tsdf_fallback"),
    "b6": ("b6_refinement_effect", "bench.exp_b6_refinement_effect"),
    "b7": ("b7_determinism", "bench.exp_b7_determinism"),
    "b8": ("b8_exif_intrinsics", "bench.exp_b8_exif_intrinsics"),
}

FIRST_SCENE = manifest["scenes"][0]["name"]

SMOKE_ARGS = {
    # B1 is the only experiment that emits a rigged GLB, so the smoke lane
    # rigs too -- capped to a handful of frames to keep it to minutes.
    "b1": [
        "--scenes",
        FIRST_SCENE,
        "--n-images",
        str(B1_SMOKE_IMAGES),
        "--articulation",
        ARTICULATION,
    ],
    "b2": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b3": ["--scenes", FIRST_SCENE, "--frame-counts", "8"],
    "b4": ["--scenes", FIRST_SCENE, "--no-budget-sweep"],
    "b5": ["--scenes", FIRST_SCENE, "--frame-counts", "8", "--thresholds", "0.2"],
    "b6": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b7": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b8": ["--scenes", FIRST_SCENE, "--n-images", "4"],
}

# Full lane: module defaults, every scene. Override here if the protocol changed.
# B1 keeps every frame so the pipeline's own MAX_RECONSTRUCTION_FRAMES=40 cap
# is the thing doing the selecting -- that behaviour is what A5/B4 are about.
FULL_ARGS = {key: [] for key in MODULES}
FULL_ARGS["b1"] = ["--articulation", ARTICULATION]

# b1 takes --deliverables-root; b2..b8 take --output-root. Both keep the
# ~1.5 GB per-run alignment cache off the persisted output volume.
OUTPUT_FLAG = {"b1": "--deliverables-root"}


def build_argv(exp, scope):
    _, module = MODULES[exp]
    args = (SMOKE_ARGS if scope == "smoke" else FULL_ARGS)[exp]
    out_root = WORK_DIR / "runs" / exp
    out_root.mkdir(parents=True, exist_ok=True)
    return [
        sys.executable,
        "-u",
        "-m",
        module,
        "--manifest",
        str(MANIFEST_PATH),
        "--results-dir",
        str(RESULTS_DIR),
        "--seed",
        "0",
        "--verbose",
        OUTPUT_FLAG.get(exp, "--output-root"),
        str(out_root),
        *args,
    ]


for exp in EXPERIMENTS:
    print(exp, " ".join(str(a) for a in build_argv(exp, SCOPE)[3:]))

## 13. Runner

One subprocess per experiment. A CUDA OOM, an OOM-killer `SIGKILL` or a VTK
segfault takes down that experiment only; the notebook records it and moves on.
`run_summary.json` is rewritten after each one, so a session killed at the wall
clock still leaves a complete account.

In [ ]:
SUMMARY_PATH = OUT_DIR / "run_summary.json"
RUNS = []
SESSION_START_ISO = _dt.datetime.now().isoformat(timespec="seconds")


def _save_summary():
    payload = {
        "run_id": RUN_ID,
        "platform": PLATFORM,
        "commit": COMMIT,
        "scope": SCOPE,
        "curope_built": CUROPE_BUILT,
        "started": SESSION_START_ISO,
        "updated": _dt.datetime.now().isoformat(timespec="seconds"),
        "runs": RUNS,
    }
    SUMMARY_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def _free_gb(path):
    return shutil.disk_usage(path).free / 1024**3


def _csv_rows(path):
    with path.open(encoding="utf-8") as fh:
        return max(0, sum(1 for _ in fh) - 1)


def run_experiment(exp, scope=None, force=None):
    scope = scope or SCOPE
    force = FORCE_RERUN if force is None else force
    exp_id, _ = MODULES[exp]
    csv_path = RESULTS_DIR / (exp_id + ".csv")
    log_path = LOG_DIR / (exp + ".log")

    if csv_path.exists() and not force:
        log.info("SKIP  %s -- %s exists (set FORCE_RERUN=True to redo)", exp, csv_path.name)
        record = {
            "exp": exp,
            "exp_id": exp_id,
            "status": "skipped",
            "minutes": 0.0,
            "rows": _csv_rows(csv_path),
            "csv": str(csv_path),
            "log": str(log_path),
        }
        RUNS.append(record)
        _save_summary()
        return record

    argv = build_argv(exp, scope)
    timeout_s = TIMEOUT_MIN[scope][exp] * 60
    log.info("=" * 70)
    log.info("START %s (%s lane, timeout %d min)", exp, scope, TIMEOUT_MIN[scope][exp])
    log.info("      disk free: work %.1f GB, out %.1f GB", _free_gb(WORK_DIR), _free_gb(OUT_DIR))
    log.info("      log -> %s", log_path)

    started = time.monotonic()
    record = {
        "exp": exp,
        "exp_id": exp_id,
        "scope": scope,
        "argv": [str(a) for a in argv],
        "started": _dt.datetime.now().isoformat(timespec="seconds"),
        "log": str(log_path),
    }
    try:
        rc, _tail = sh(
            argv,
            cwd=REPO_DIR,
            env=CHILD_ENV,
            log_path=log_path,
            timeout_s=timeout_s,
            check=True,
            echo=True,
        )
        record["status"] = "ok"
        record["returncode"] = rc
    except CommandFailed as exc:
        record["status"] = "timeout" if exc.timed_out else "failed"
        record["returncode"] = exc.returncode
        record["error"] = exc.reason
        record["tail"] = exc.tail
        log.error("FAIL  %s: %s", exp, exc.reason)
        log.error("      last lines:\n%s", exc.tail)
    except Exception as exc:  # the runner itself broke
        record["status"] = "error"
        record["error"] = repr(exc)
        record["traceback"] = traceback.format_exc()
        log.exception("ERROR %s: runner failed", exp)

    record["minutes"] = round((time.monotonic() - started) / 60, 2)
    if csv_path.exists():
        record["csv"] = str(csv_path)
        record["rows"] = _csv_rows(csv_path)
        log.info("      %s: %d rows -> %s", exp, record["rows"], csv_path.name)
        if record["rows"] == 0:
            log.warning("      %s produced ZERO rows -- check gt_path and --scenes", exp)
    else:
        record["csv"] = None
        record["rows"] = 0
        if record["status"] == "ok":
            log.warning("      %s exited 0 but wrote no CSV", exp)

    log.info("DONE  %s: %s in %.1f min", exp, record["status"], record["minutes"])
    RUNS.append(record)
    _save_summary()

    if CLEAN_WORK_AFTER_EACH:
        target = WORK_DIR / "runs" / exp
        if target.exists():
            size_gb = sum(f.stat().st_size for f in target.rglob("*") if f.is_file()) / 1024**3
            shutil.rmtree(target, ignore_errors=True)
            log.info("      cleaned %.1f GB from %s", size_gb, target)
    return record

## 14. Run the tier

Stops launching new experiments once `SESSION_BUDGET_MIN` is reached, rather
than starting something the platform's wall clock will cut off. Re-run the
notebook to pick up where it stopped.

In [ ]:
SESSION_START = time.monotonic()
SESSION_START_ISO = _dt.datetime.now().isoformat(timespec="seconds")
RUNS.clear()

log.info("Tier B: %s lane, %d experiments, commit %s", SCOPE, len(EXPERIMENTS), COMMIT[:8])

for exp in EXPERIMENTS:
    elapsed_min = (time.monotonic() - SESSION_START) / 60
    if elapsed_min > SESSION_BUDGET_MIN:
        log.warning(
            "session budget reached (%.0f min); not launching %s. Re-run the "
            "notebook to continue -- finished experiments are skipped.",
            elapsed_min,
            exp,
        )
        RUNS.append(
            {
                "exp": exp,
                "status": "not_started",
                "minutes": 0.0,
                "rows": 0,
                "reason": "session budget",
            }
        )
        _save_summary()
        continue
    run_experiment(exp)

print()
print("=" * 78)
print(f"{'exp':<5} {'status':<12} {'min':>7} {'rows':>6}  log")
print("-" * 78)
for r in RUNS:
    print(
        f"{r['exp']:<5} {r.get('status', '?'):<12} {r.get('minutes', 0.0):>7.1f} "
        f"{r.get('rows', 0):>6}  {Path(r.get('log', '-')).name}"
    )
print("=" * 78)

failed = [r for r in RUNS if r.get("status") in ("failed", "error", "timeout")]
print(f"\n{len(RUNS) - len(failed)}/{len(RUNS)} ok; summary -> {SUMMARY_PATH}")
for r in failed:
    print(f"\n--- {r['exp']}: {r.get('error', '')} ---")
    print(r.get("tail") or r.get("traceback") or "(see log)")

## 15. Collect results

Everything the paper needs, zipped into one file: the CSVs, every log, and the
run summary. On Kaggle the zip appears under the notebook's Output tab; on Colab
it lands in Drive if you mounted it.

In [ ]:
bundle = OUT_DIR / ("tier_b_" + RUN_ID)
bundle.mkdir(parents=True, exist_ok=True)
(bundle / "results").mkdir(exist_ok=True)

for csv in RESULTS_DIR.glob("*.csv"):
    shutil.copy2(csv, bundle / "results" / csv.name)
shutil.copytree(LOG_DIR, bundle / "logs", dirs_exist_ok=True)
if SUMMARY_PATH.exists():
    shutil.copy2(SUMMARY_PATH, bundle / "run_summary.json")

(bundle / "environment.txt").write_text(
    "\n".join(
        [
            "run_id      " + RUN_ID,
            "platform    " + PLATFORM,
            "commit      " + COMMIT,
            "scope       " + SCOPE,
            "python      " + platform.python_version(),
            "os          " + platform.platform(),
            "curope      " + ("compiled" if CUROPE_BUILT else "pytorch fallback"),
        ]
    ),
    encoding="utf-8",
)

archive = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
log.info("bundle -> %s (%.1f MB)", archive, Path(archive).stat().st_size / 1024**2)

print("\nCSVs produced:")
for csv in sorted((bundle / "results").glob("*.csv")):
    print(f"  {csv.name:<44} {_csv_rows(csv):>6} rows  {csv.stat().st_size / 1024:.0f} KB")

print("\nTo bring these home: download the zip, unpack into bench/results/, then")
print("  git add bench/results && git commit -m 'bench: Tier B results'")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `exit -9 / SIGKILL` | host RAM OOM killer, not VRAM | lower `--n-images`, or `--image-size 384` |
| status `timeout` | exceeded `TIMEOUT_MIN[scope][exp]` | raise the ceiling if the run was genuinely progressing; the log shows where it stalled |
| `torch.cuda.OutOfMemoryError` | VRAM | lower `--image-size`; for B3/B5 lower `--frame-counts` |
| `exit -11 / SIGSEGV` | native crash in VTK or CUDA | read `logs/<run_id>/faulthandler.log` for the Python frames |
| `no scenes` SystemExit | nothing attached, or `DATA_ROOT` is wrong | Kaggle sidebar > Input > Add Input, then point `DATA_ROOT` at the mount |
| dataset test `decode` FAIL | corrupt or unusual bit depth | drop the file, or convert the scene to 8-bit RGB |
| dataset test `ground_truth` FAIL | trimesh cannot open the GT | check it is a real `.ply`/`.obj` and not a Git-LFS pointer or an archive |
| dataset test `tau_vs_gt_scale` WARN | manifest units disagree with the GT file | set `tau`/`units` per scene: 2 mm is DTU's, a COLMAP cloud is in scene units |
| discovery names a scene `Rectified` | the images sit in a folder not in `IMAGE_DIR_NAMES` | add that folder name to `IMAGE_DIR_NAMES`, or list the scene in `SCENES` |
| exits 0, **zero rows** | scene has `gt_path: null`, or `--scenes` matched nothing | scene names must match the manifest exactly |
| B4's three variants identical | scene has <= 40 frames, so the budget never fires | use a real extracted video sequence |
| `MASt3R is not installed` | cell 7 did not finish, or the kernel restarted | re-run cell 7; `--no-deps` means torch is never touched |
| `numpy.dtype size changed` | numpy 2 vs a wheel built for numpy 1 | set `PIN_NUMPY_LT2 = True`, restart the kernel, re-run from cell 2 |
| session killed at 12 h | platform wall clock | re-run; completed CSVs are skipped |
| `disk quota exceeded` on Kaggle | reconstructions landing in `/kaggle/working` | confirm `--output-root` points into `/kaggle/temp` |

**For the paper:** run every Tier B row on one GPU type. `env_metadata()` stamps
`gpu`, `torch` and `git_commit` onto every row, so mixed hardware is at least
detectable after the fact - but it is not comparable, and B7's determinism
result in particular is meaningless across devices.